# Customers Silver Layer

In [1]:
from pyspark.sql import SparkSession

In [2]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName("blue-owls-assessment") \
    .getOrCreate()

In [8]:
customers = spark.read.format('csv').option('inferSchema', True).option('header', True).load('output/bronze/customers')

In [12]:
customers.describe()

DataFrame[summary: string, customer_city: string, customer_id: string, customer_state: string, customer_unique_id: string, customer_zip_code_prefix: string, _source_endpoint: string]

In [30]:
from pyspark.sql.functions import *

In [31]:
customers.show()

+----------------+--------------------+--------------+--------------------+------------------------+--------------------+--------------------+
|   customer_city|         customer_id|customer_state|  customer_unique_id|customer_zip_code_prefix|        _ingested_at|    _source_endpoint|
+----------------+--------------------+--------------+--------------------+------------------------+--------------------+--------------------+
|     jaboticabal|f63d914e794b8fbcd...|            SP|030e8f4f620aaddba...|                   14875|2026-04-01 05:36:...|/api/v1/data/cust...|
|          uruacu|5e152b0feca6b29b0...|            GO|c91a7e9c7335c7af4...|                   76400|2026-04-01 05:36:...|/api/v1/data/cust...|
|  rio de janeiro|afef906b644b83715...|            RJ|16d25bcacd8f60bc1...|                   22795|2026-04-01 05:36:...|/api/v1/data/cust...|
|  rio de janeiro|5bee899f25d069092...|            RJ|b947b8b75b0ba7db5...|                   22790|2026-04-01 05:36:...|/api/v1/data/cust...|

In [32]:
customers_clean = customers.select(
    trim(lower('customer_city')).alias('customer_city'),
    col('customer_id'),
    trim(upper('customer_state')).alias('customer_state'),
    col('customer_unique_id'),
    col('customer_zip_code_prefix').cast('int').alias('customer_zip_code_prefix'),
    col('_ingested_at').cast('timestamp').alias('_ingested_at'),
    col('_source_endpoint')
)

In [33]:
customers_clean.describe()

DataFrame[summary: string, customer_city: string, customer_id: string, customer_state: string, customer_unique_id: string, customer_zip_code_prefix: string, _source_endpoint: string]

In [34]:
customers_clean.show()

+----------------+--------------------+--------------+--------------------+------------------------+--------------------+--------------------+
|   customer_city|         customer_id|customer_state|  customer_unique_id|customer_zip_code_prefix|        _ingested_at|    _source_endpoint|
+----------------+--------------------+--------------+--------------------+------------------------+--------------------+--------------------+
|     jaboticabal|f63d914e794b8fbcd...|            SP|030e8f4f620aaddba...|                   14875|2026-04-01 05:36:...|/api/v1/data/cust...|
|          uruacu|5e152b0feca6b29b0...|            GO|c91a7e9c7335c7af4...|                   76400|2026-04-01 05:36:...|/api/v1/data/cust...|
|  rio de janeiro|afef906b644b83715...|            RJ|16d25bcacd8f60bc1...|                   22795|2026-04-01 05:36:...|/api/v1/data/cust...|
|  rio de janeiro|5bee899f25d069092...|            RJ|b947b8b75b0ba7db5...|                   22790|2026-04-01 05:36:...|/api/v1/data/cust...|

In [35]:
customers_clean = customers_clean.fillna(
    {
        'customer_city': 'unknown',
        'customer_state': 'NA'
    }
)

In [39]:
customers_validated = customers_clean.withColumn(
    '_is_valid',
    when(
        (col('customer_id').isNull()) |
        (col('customer_unique_id').isNull()) |
        (col('customer_zip_code_prefix').isNull()) |
        (col('customer_zip_code_prefix')<=0),
        lit(False)
    ).otherwise(lit(True))
)

In [46]:
from pyspark.sql import Window

In [47]:
window_spec = Window.partitionBy(col('customer_id')).orderBy(col('_ingested_at').desc())

In [49]:
from pyspark.sql.functions import row_number

In [54]:
customers_dedup = customers_validated.withColumn(
    'row_num', row_number().over(window_spec)
).filter(
    col('row_num')==1
).drop(
    'row_num'
)

In [56]:
customers_dedup.coalesce(1).write \
    .mode("overwrite") \
    .option("header", True) \
    .csv("output/silver/customers")

# Orders silver layer

In [58]:
orders = spark.read.format('csv').option('inferSchema', True).option('header', True).load('output/bronze/orders')

In [59]:
orders.show()

+--------------------+-------------------+----------------------------+-----------------------------+-----------------------------+--------------------+------------------------+------------+--------------------+-------------------+
|         customer_id|  order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|            order_id|order_purchase_timestamp|order_status|        _ingested_at|   _source_endpoint|
+--------------------+-------------------+----------------------------+-----------------------------+-----------------------------+--------------------+------------------------+------------+--------------------+-------------------+
|3b2a2167b75d2d89c...|2017-05-01 11:35:19|         2017-05-02 21:25:52|          2017-05-10 05:18:34|          2017-05-25 00:00:00|894e3de9f4244f511...|     2017-05-01 11:28:31|   delivered|2026-04-01 05:35:...|/api/v1/data/orders|
|4b87e315da3b2afd6...|2018-07-07 10:30:11|         2018-07-16 13:03:00| 

In [63]:
orders_clean = orders.select(
    col('customer_id'),
    col('order_id'),
    col('order_status').alias('order_status'),
    col('order_purchase_timestamp').cast('timestamp'),
    col('order_approved_at').cast('timestamp'),
    col('order_delivered_carrier_date').cast('timestamp'),
    col('order_delivered_customer_date').cast('timestamp'),
    col('order_estimated_delivery_date').cast('timestamp'),
    col('_ingested_at').cast('timestamp'),
    col('_source_endpoint')
).withColumn(
    'order_status', lower(trim(col('order_status')))
)

In [65]:
orders_clean.show()

+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+--------------------+-------------------+
|         customer_id|            order_id|order_status|order_purchase_timestamp|  order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|        _ingested_at|   _source_endpoint|
+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+--------------------+-------------------+
|3b2a2167b75d2d89c...|894e3de9f4244f511...|   delivered|     2017-05-01 11:28:31|2017-05-01 11:35:19|         2017-05-02 21:25:52|          2017-05-10 05:18:34|          2017-05-25 00:00:00|2026-04-01 05:35:...|/api/v1/data/orders|
|4b87e315da3b2afd6...|c87c652834da18e71...|   delivered|     2018-07-07 

In [66]:
orders_validated = orders_clean.withColumn(
    '_is_valid',
    when(
        (col('order_id').isNull()) |
        (col('customer_id').isNull()) |
        (col('order_delivered_customer_date') < col('order_purchase_timestamp')) |
        (col('order_approved_at') < col('order_purchase_timestamp')),
        lit(False)
    ).otherwise(
        lit(True)
    )
)

In [68]:
window_spec = Window.partitionBy(col('order_id')).orderBy(col('_ingested_at').desc())

In [69]:
orders_dedupd = orders_validated.withColumn(
    'row_num', row_number().over(window_spec)
).filter(col('row_num')==1).drop(col('row_num'))

In [70]:
orders_dedupd.show()

+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+--------------------+-------------------+---------+
|         customer_id|            order_id|order_status|order_purchase_timestamp|  order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|        _ingested_at|   _source_endpoint|_is_valid|
+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+--------------------+-------------------+---------+
|f6dd3ec061db4e398...|00018f77f2f0320c5...|   delivered|     2017-04-26 10:53:06|2017-04-26 11:05:13|         2017-05-04 14:35:00|          2017-05-12 16:04:24|          2017-05-15 00:00:00|2026-04-01 05:35:...|/api/v1/data/orders|     true|
|58dbd0b2d70206bf4...|00042b26cf

In [71]:
orders_dedupd.coalesce(1).write.mode(
    'overwrite'
).option('header', True).csv(
    'output/silver/orders'
)

# Order Items Silver Layer

In [72]:
order_items = spark.read.format('csv').option('inferSchema', True).option('header', True).load('output/bronze/order_items')

In [73]:
order_items.show()

+-------------+--------------------+-------------+------+--------------------+--------------------+-------------------+--------------------+--------------------+
|freight_value|            order_id|order_item_id| price|          product_id|           seller_id|shipping_limit_date|        _ingested_at|    _source_endpoint|
+-------------+--------------------+-------------+------+--------------------+--------------------+-------------------+--------------------+--------------------+
|        15.79|1af38227c08898011...|            1| 147.9|060cb19345d90064d...|8581055ce74af1dab...|2017-08-30 15:55:19|2026-04-01 05:51:...|/api/v1/data/orde...|
|        14.26|1af3db74a164e8f5a...|            1|  94.0|30ac6df06dc59ad72...|0dd184061fb0eaa7c...|2018-01-19 19:30:46|2026-04-01 05:51:...|/api/v1/data/orde...|
|        16.11|1af47f4805c833e81...|            1|  29.5|44a34214a57dc373d...|7008613ea464bad5c...|2017-06-21 03:10:41|2026-04-01 05:51:...|/api/v1/data/orde...|
|        16.11|1af47f4805c83

In [75]:
orders.show()

+--------------------+-------------------+----------------------------+-----------------------------+-----------------------------+--------------------+------------------------+------------+--------------------+-------------------+
|         customer_id|  order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|            order_id|order_purchase_timestamp|order_status|        _ingested_at|   _source_endpoint|
+--------------------+-------------------+----------------------------+-----------------------------+-----------------------------+--------------------+------------------------+------------+--------------------+-------------------+
|3b2a2167b75d2d89c...|2017-05-01 11:35:19|         2017-05-02 21:25:52|          2017-05-10 05:18:34|          2017-05-25 00:00:00|894e3de9f4244f511...|     2017-05-01 11:28:31|   delivered|2026-04-01 05:35:...|/api/v1/data/orders|
|4b87e315da3b2afd6...|2018-07-07 10:30:11|         2018-07-16 13:03:00| 

In [76]:
order_items_clean = order_items.select(
    col('order_id'),
    col('order_item_id').cast('int'),
    col('product_id'),
    col('seller_id'),
    col('price').cast('double'),
    col('freight_value').cast('double'),
    col('shipping_limit_date').cast('timestamp'),
    col('_ingested_at').cast('timestamp'),
    col('_source_endpoint')
)

In [77]:
order_items_clean_with_fk = order_items_clean.join(
    orders.select('order_id').withColumnRenamed(
        'order_id', 'valid_order_id'
    ),
    order_items_clean.order_id==col('valid_order_id'),
    'left'
)

In [78]:
order_items_clean_with_fk.show()

+--------------------+-------------+--------------------+--------------------+------+-------------+-------------------+--------------------+--------------------+--------------------+
|            order_id|order_item_id|          product_id|           seller_id| price|freight_value|shipping_limit_date|        _ingested_at|    _source_endpoint|      valid_order_id|
+--------------------+-------------+--------------------+--------------------+------+-------------+-------------------+--------------------+--------------------+--------------------+
|1af38227c08898011...|            1|060cb19345d90064d...|8581055ce74af1dab...| 147.9|        15.79|2017-08-30 15:55:19|2026-04-01 05:51:...|/api/v1/data/orde...|1af38227c08898011...|
|1af3db74a164e8f5a...|            1|30ac6df06dc59ad72...|0dd184061fb0eaa7c...|  94.0|        14.26|2018-01-19 19:30:46|2026-04-01 05:51:...|/api/v1/data/orde...|1af3db74a164e8f5a...|
|1af47f4805c833e81...|            1|44a34214a57dc373d...|7008613ea464bad5c...|  29.5|

In [80]:
order_items_clean_with_fk_validated = order_items_clean_with_fk.withColumn(
    '_is_valid',
    when(
        (col('order_id').isNull()) |
        (col('product_id').isNull()) |
        (col('seller_id').isNull()) |
        (col('price')<0) |
        (col('freight_value')<0) |
        (col('order_item_id')<=0) |
        (col('valid_order_id').isNull()),
        lit(False)
    ).otherwise(lit(True))
).drop('valid_order_id')

In [81]:
order_items_clean_with_fk_validated.show()

+--------------------+-------------+--------------------+--------------------+------+-------------+-------------------+--------------------+--------------------+---------+
|            order_id|order_item_id|          product_id|           seller_id| price|freight_value|shipping_limit_date|        _ingested_at|    _source_endpoint|_is_valid|
+--------------------+-------------+--------------------+--------------------+------+-------------+-------------------+--------------------+--------------------+---------+
|1af38227c08898011...|            1|060cb19345d90064d...|8581055ce74af1dab...| 147.9|        15.79|2017-08-30 15:55:19|2026-04-01 05:51:...|/api/v1/data/orde...|     true|
|1af3db74a164e8f5a...|            1|30ac6df06dc59ad72...|0dd184061fb0eaa7c...|  94.0|        14.26|2018-01-19 19:30:46|2026-04-01 05:51:...|/api/v1/data/orde...|     true|
|1af47f4805c833e81...|            1|44a34214a57dc373d...|7008613ea464bad5c...|  29.5|        16.11|2017-06-21 03:10:41|2026-04-01 05:51:...|

In [83]:
window_spec = Window.partitionBy("order_id", "order_item_id") \
                    .orderBy(col("_ingested_at").desc())

In [84]:
order_items_clean_with_fk_validated_dedupd = order_items_clean_with_fk_validated.withColumn(
    "row_num", row_number().over(window_spec)
).filter(
    col("row_num") == 1
).drop("row_num")

In [85]:
order_items_clean_with_fk_validated_dedupd.coalesce(1).write \
    .mode("overwrite") \
    .option("header", True) \
    .csv("output/silver/order_items")

# Payments Silver Layer

In [86]:
payments = spark.read.format('csv').option('inferSchema', True).option('header', True).load('output/bronze/payments')

In [87]:
payments.show()

+--------------------+--------------------+------------------+------------+-------------+--------------------+--------------------+
|            order_id|payment_installments|payment_sequential|payment_type|payment_value|        _ingested_at|    _source_endpoint|
+--------------------+--------------------+------------------+------------+-------------+--------------------+--------------------+
|7a8f4f62037bb98af...|                   3|                 1| credit_card|       186.16|2026-04-01 05:53:...|/api/v1/data/paym...|
|9eecf4b8716283fcb...|                   1|                 1|      boleto|       359.91|2026-04-01 05:53:...|/api/v1/data/paym...|
|1800254c5ad71baad...|                   6|                 1| credit_card|       299.98|2026-04-01 05:53:...|/api/v1/data/paym...|
|d6a544caf1fd9274c...|                   1|                 1| credit_card|        28.89|2026-04-01 05:53:...|/api/v1/data/paym...|
|f4efba146388fcb18...|                   1|                 1| credit_card| 

In [89]:
orders.show()

+--------------------+-------------------+----------------------------+-----------------------------+-----------------------------+--------------------+------------------------+------------+--------------------+-------------------+
|         customer_id|  order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|            order_id|order_purchase_timestamp|order_status|        _ingested_at|   _source_endpoint|
+--------------------+-------------------+----------------------------+-----------------------------+-----------------------------+--------------------+------------------------+------------+--------------------+-------------------+
|3b2a2167b75d2d89c...|2017-05-01 11:35:19|         2017-05-02 21:25:52|          2017-05-10 05:18:34|          2017-05-25 00:00:00|894e3de9f4244f511...|     2017-05-01 11:28:31|   delivered|2026-04-01 05:35:...|/api/v1/data/orders|
|4b87e315da3b2afd6...|2018-07-07 10:30:11|         2018-07-16 13:03:00| 

In [90]:
payments_clean = payments.select(
    col('order_id'),
    col('payment_installments').cast('int'),
    col('payment_sequential').cast('int'),
    col('payment_type'),
    col('payment_value').cast('double'),
    col('_ingested_at').cast('timestamp'),
    col('_source_endpoint')
).withColumn(
    'payment_type', lower(trim(col('payment_type')))
)

In [92]:
payments_clean_with_fk = payments_clean.join(
    orders.select('order_id').withColumnRenamed('order_id', 'valid_order_id'),
    payments_clean.order_id==col('valid_order_id'),
    'left'
)

In [93]:
payments_clean_with_fk_validated = payments_clean_with_fk.withColumn(
    '_is_valid',
    when(
        (col('order_id').isNull()) |
        (col('payment_value') <= 0) |
        (col('payment_installments') <= 0) |
        (col('payment_sequential') <= 0) |
        (col('payment_type').isNull()) |
        (col('valid_order_id').isNull()),
        lit(False)
    ).otherwise(True)
).drop(col('valid_order_id'))

In [96]:
window_spec = Window.partitionBy("order_id", "payment_sequential").orderBy(col("_ingested_at").desc())

In [97]:
payments_clean_with_fk_validated_dedupd = payments_clean_with_fk_validated.withColumn("row_num", row_number().over(window_spec)) \
                        .filter(col("row_num") == 1) \
                        .drop("row_num")

In [98]:
payments_clean_with_fk_validated_dedupd.coalesce(1).write \
    .mode("overwrite") \
    .option("header", True) \
    .csv("output/silver/payments")

# Products Silver Layer

In [99]:
products = spark.read.format('csv').option('inferSchema', True).option('header', True).load('output/bronze/products')

In [100]:
products.show()

+---------------------+--------------------------+-----------------+--------------------+-----------------+-------------------+------------------+----------------+----------------+--------------------+--------------------+
|product_category_name|product_description_lenght|product_height_cm|          product_id|product_length_cm|product_name_lenght|product_photos_qty|product_weight_g|product_width_cm|        _ingested_at|    _source_endpoint|
+---------------------+--------------------------+-----------------+--------------------+-----------------+-------------------+------------------+----------------+----------------+--------------------+--------------------+
|         beleza_saude|                       362|                5|fdf37ff6a4ff39ac9...|               27|                 38|                 2|             300|              16|2026-04-01 05:53:...|/api/v1/data/prod...|
|     eletrodomesticos|                       133|               13|5acad00c7ade065ca...|               27| 

In [101]:
products_clean = products.select(
    col("product_id"),
    trim(lower(col("product_category_name"))).alias("product_category_name"),
    
    col("product_description_lenght").cast("int"),
    col("product_name_lenght").cast("int"),
    col("product_photos_qty").cast("int"),
    
    col("product_weight_g").cast("double"),
    col("product_length_cm").cast("double"),
    col("product_height_cm").cast("double"),
    col("product_width_cm").cast("double"),
    
    col("_ingested_at").cast("timestamp"),
    col("_source_endpoint")
)

In [102]:
products_clean = products_clean.fillna({
    "product_category_name": "unknown"
})

In [103]:
products_clean_validated = products_clean.withColumn(
    "_is_valid",
    when(
        (col("product_id").isNull()) |
        (col("product_weight_g") <= 0) |
        (col("product_length_cm") <= 0) |
        (col("product_height_cm") <= 0) |
        (col("product_width_cm") <= 0) |
        (col("product_photos_qty") < 0),
        lit(False)
    ).otherwise(lit(True))
)

In [104]:
window_spec = Window.partitionBy("product_id").orderBy(col("_ingested_at").desc())

In [105]:
products_clean_validated_deduped = products_clean_validated.withColumn(
    "row_num", row_number().over(window_spec)
).filter(
    col("row_num") == 1
).drop("row_num")

In [106]:
products_clean_validated_deduped.coalesce(1).write.mode("overwrite").option("header", True).csv("output/silver/products")

# Sellers Silver Layer

In [107]:
sellers = spark.read.format('csv').option('inferSchema', True).option('header', True).load('output/bronze/sellers')

In [108]:
sellers.show()

+--------------------+--------------------+------------+----------------------+--------------------+--------------------+
|         seller_city|           seller_id|seller_state|seller_zip_code_prefix|        _ingested_at|    _source_endpoint|
+--------------------+--------------------+------------+----------------------+--------------------+--------------------+
|      belo horizonte|bafc5aa11e413978b...|          MG|                 30130|2026-04-01 05:53:...|/api/v1/data/sellers|
|          santa rosa|44717f64ec2a45797...|          RS|                 98780|2026-04-01 05:53:...|/api/v1/data/sellers|
|         curitibanos|b7cc6c5e001441ae8...|          SC|                 89520|2026-04-01 05:53:...|/api/v1/data/sellers|
|   laranjal paulista|af7cdac9d9897cdde...|          SP|                 18500|2026-04-01 05:53:...|/api/v1/data/sellers|
|      ribeirao preto|b3e2faf2c1004d674...|          SP|                 14093|2026-04-01 05:53:...|/api/v1/data/sellers|
|      ribeirao preto|67

In [109]:
sellers_clean = sellers.select(
    col("seller_id"),
    trim(lower(col("seller_city"))).alias("seller_city"),
    trim(upper(col("seller_state"))).alias("seller_state"),
    col("seller_zip_code_prefix").cast("int"),
    col("_ingested_at").cast("timestamp"),
    col("_source_endpoint")
)

In [110]:
sellers_clean = sellers_clean.fillna({
    "seller_city": "unknown",
    "seller_state": "NA"
})

In [111]:
sellers_clean_validated = sellers_clean.withColumn(
    "_is_valid",
    when(
        (col("seller_id").isNull()) |
        (col("seller_zip_code_prefix").isNull()) |
        (col("seller_zip_code_prefix") <= 0),
        lit(False)
    ).otherwise(lit(True))
)

In [112]:
window_spec = Window.partitionBy("seller_id").orderBy(col("_ingested_at").desc())

In [113]:
sellers_clean_validated_dedupd = sellers_clean_validated.withColumn(
    "row_num", row_number().over(window_spec)
).filter(
    col("row_num") == 1
).drop("row_num")

In [114]:
sellers_clean_validated_dedupd.coalesce(1).write.mode("overwrite").option("header", True).csv("output/silver/sellers")